# Data Pipeline

First notebook in the split workflow. It parses the MediaWiki XML, cleans text, builds candidate relationship examples, chooses either weak labels or an externally generated LLM Judge label file, and writes the shared train/dev/test files under `data/`. The LLM Judge itself lives in `01.5_llm_judge_colab.ipynb`.


## 1. Configuration

In [1]:
from pathlib import Path

# -----------------------------------------------------------------------------
# Shared data folder.
# -----------------------------------------------------------------------------
# All files that are inputs to, outputs from, or shared between multiple models
# are stored in data/. Model-specific artifacts remain in their own folders.
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Input XML.
XML_PATH = DATA_DIR / "bookworm_09062026.xml"

# Shared preprocessing and training files used by the split workflow.
PAGES_CSV = DATA_DIR / "pages.csv"
CHARACTER_GAZETTEER_CSV = DATA_DIR / "character_gazetteer.csv"
CANDIDATE_EXAMPLES_CSV = DATA_DIR / "candidate_examples.csv"
LLM_LABELED_CANDIDATES_CSV = DATA_DIR / "candidate_examples_llm_labeled.csv"
WEAK_LABEL_DISTRIBUTION_CSV = DATA_DIR / "label_distribution.csv"
TRAINING_LABEL_DISTRIBUTION_CSV = DATA_DIR / "training_label_distribution.csv"
TRAIN_CSV = DATA_DIR / "train.csv"
DEV_CSV = DATA_DIR / "dev.csv"
TEST_CSV = DATA_DIR / "test.csv"
SPLIT_LABEL_DISTRIBUTION_CSV = DATA_DIR / "split_label_distribution.csv"
PIPELINE_RUN_METADATA_JSON = DATA_DIR / "pipeline_run_metadata.json"

# Model-specific output folders.
BASELINE_DIR = Path("baseline")  # TF-IDF + Logistic Regression artifacts.
BILSTM_DIR = Path("bilstm")      # BiLSTM artifacts.
BASELINE_DIR.mkdir(parents=True, exist_ok=True)
BILSTM_DIR.mkdir(parents=True, exist_ok=True)

# Relationship labels. "no_relation" is needed as the negative class.
RELATIONSHIPS = [
    "family",
    "romantic",
    "friend_ally",
    "service_retainer",
    "enemy_rival",
    "no_relation",
]

NO_RELATION_LABEL = "no_relation"

RANDOM_SEED = 42
TEST_SIZE = 0.15
DEV_SIZE = 0.15
MAX_NEGATIVE_RATIO = 2.0
MIN_CONTEXT_CHARS = 25

print(f"XML path: {XML_PATH.resolve()}")
print(f"Shared data folder: {DATA_DIR.resolve()}")
print(f"Baseline output folder: {BASELINE_DIR.resolve()}")
print(f"BiLSTM output folder: {BILSTM_DIR.resolve()}")
print(f"Relationship labels: {RELATIONSHIPS}")

# -----------------------------------------------------------------------------
# External labeling handoff.
# -----------------------------------------------------------------------------
# The LLM Judge itself is kept in 01_llm_judge_colab_gpu_minimal.ipynb.
# This notebook only consumes its output when the file exists.
# Options: "auto", "weak_labels", "llm_judge".
# - auto: use data/candidate_examples_llm_labeled.csv if present, otherwise weak labels.
# - weak_labels: always use data/candidate_examples.csv.
# - llm_judge: require data/candidate_examples_llm_labeled.csv.
LABEL_SOURCE_MODE = "auto"

print(f"Label source mode: {LABEL_SOURCE_MODE}")
print(f"Candidate examples CSV: {CANDIDATE_EXAMPLES_CSV}")
print(f"Optional external LLM-labeled CSV: {LLM_LABELED_CANDIDATES_CSV}")


XML path: E:\Natural Language Processing\Project 2\data\bookworm_09062026.xml
Shared data folder: E:\Natural Language Processing\Project 2\data
Baseline output folder: E:\Natural Language Processing\Project 2\baseline
BiLSTM output folder: E:\Natural Language Processing\Project 2\bilstm
Relationship labels: ['family', 'romantic', 'friend_ally', 'service_retainer', 'enemy_rival', 'no_relation']
Label source mode: auto
Candidate examples CSV: data\candidate_examples.csv
Optional external LLM-labeled CSV: data\candidate_examples_llm_labeled.csv


## 2. Imports

In [2]:
import html
import json
import re
import warnings
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit, train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 140)
np.random.seed(RANDOM_SEED)


## 3. Parse the MediaWiki XML

MediaWiki exports store the page title and page text as XML elements. This cell extracts them as:

- `character_name` = page `<title>`
- `character_text` = revision `<text>`

Characters pages that have less than 2000 characters are removed

In [3]:
def get_xml_namespace(tag: str) -> str:
    """Return namespace prefix in ElementTree format, e.g. '{...}', or empty string."""
    if tag.startswith("{"):
        return tag.split("}", 1)[0] + "}"
    return ""


def parse_mediawiki_xml(xml_path: Path) -> pd.DataFrame:
    """Parse a MediaWiki XML export into one row per page."""
    xml_path = Path(xml_path)
    if not xml_path.exists():
        raise FileNotFoundError(f"XML file not found: {xml_path}")

    pages = []
    context = ET.iterparse(str(xml_path), events=("start", "end"))
    _, root = next(context)
    ns = get_xml_namespace(root.tag)

    for event, elem in context:
        if event == "end" and elem.tag == f"{ns}page":
            title = (elem.findtext(f"{ns}title") or "").strip()
            ns_id = (elem.findtext(f"{ns}ns") or "").strip()
            page_id = (elem.findtext(f"{ns}id") or "").strip()
            revision = elem.find(f"{ns}revision")
            text = ""
            timestamp = ""
            if revision is not None:
                text = revision.findtext(f"{ns}text") or ""
                timestamp = revision.findtext(f"{ns}timestamp") or ""

            is_redirect = elem.find(f"{ns}redirect") is not None or text.lstrip().upper().startswith("#REDIRECT")

            pages.append(
                {
                    "page_id": page_id,
                    "namespace": ns_id,
                    "character_name": title,
                    "character_text": text,
                    "revision_timestamp": timestamp,
                    "is_redirect": is_redirect,
                    "text_length": len(text),
                }
            )
            elem.clear()
            root.clear()

    return pd.DataFrame(pages)


pages_df = parse_mediawiki_xml(XML_PATH)

pages_df = pages_df[
    (pages_df["namespace"] == "0")
    & (~pages_df["is_redirect"])
    & (pages_df["text_length"] >= 2000)
].copy()

pages_df = pages_df.sort_values("character_name").reset_index(drop=True)
pages_df.to_csv(PAGES_CSV, index=False)

print(f"Parsed character pages with text_length >= 2000: {len(pages_df):,}")
pages_df[["page_id", "character_name", "text_length", "revision_timestamp"]].head(10)

Parsed character pages with text_length >= 2000: 257


,page_id,character_name,text_length,revision_timestamp
0,3753,Achim,3588,2025-12-18T23:44:30Z
1,3880,Adelbert,9599,2026-03-24T02:58:11Z
2,3934,Adolphine,19043,2026-05-03T19:26:14Z
3,5956,Adrett,3808,2025-05-23T14:26:44Z
4,7262,Aeussewahl,2196,2026-03-08T01:09:51Z
5,7266,Albsenti,2682,2026-03-08T02:57:20Z
6,3836,Alexis,7005,2026-02-26T22:10:43Z
7,3615,Anastasius,6855,2026-03-12T03:17:38Z
8,6591,Andrea,2153,2026-04-13T23:37:05Z
9,1287,Angelica,8337,2026-04-18T16:50:45Z


## 4. Wikitext cleaning and character gazetteer

The character gazetteer is built from page titles. The cleaning functions remove common wiki markup while preserving readable text for sentence-level candidate generation.

In [4]:
def normalize_title(title: str) -> str:
    """Normalize a MediaWiki title for matching."""
    title = html.unescape(str(title or ""))
    title = title.replace("_", " ").strip()
    title = re.sub(r"\s+", " ", title)
    return title


def strip_anchor(title: str) -> str:
    """Remove a #section anchor from a wiki title."""
    return normalize_title(str(title).split("#", 1)[0])


def extract_wikilinks(wikitext: str):
    """Extract wiki links as (target, display_text) pairs."""
    if not isinstance(wikitext, str):
        return []
    links = []
    for match in re.finditer(r"\[\[([^\]|#]+)(?:#[^\]|]*)?(?:\|([^\]]+))?\]\]", wikitext):
        target = strip_anchor(match.group(1))
        display_text = normalize_title(match.group(2) if match.group(2) else target)
        if target:
            links.append((target, display_text))
    return links


def remove_templates(text: str) -> str:
    """Remove balanced-looking {{...}} templates iteratively.

    This is a lightweight regex cleaner for a baseline notebook. It is not a full
    MediaWiki parser, but it is enough for candidate extraction and TF-IDF text.
    """
    pattern = re.compile(r"\{\{[^{}]*\}\}", flags=re.DOTALL)
    previous = None
    while previous != text:
        previous = text
        text = pattern.sub(" ", text)
    return text


def clean_wikitext(wikitext: str) -> str:
    """Convert raw wikitext into plain-ish text suitable for TF-IDF."""
    if not isinstance(wikitext, str):
        return ""

    text = html.unescape(wikitext)
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)
    text = re.sub(r"<ref\b[^>/]*/>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<ref\b[^>]*>.*?</ref>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"<gallery\b[^>]*>.*?</gallery>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"\[\[(?:Category|File|Image):[^\]]+\]\]", " ", text, flags=re.IGNORECASE)

    # Convert headings like {{h1|Story}} or {{h2|[[Part 4 Volume 3]]}} to text before template removal.
    text = re.sub(r"\{\{h[12]\|([^}|]+).*?\}\}", lambda m: f"\n{clean_wikitext(m.group(1))}\n", text, flags=re.IGNORECASE | re.DOTALL)

    # Convert wiki links to display text.
    text = re.sub(
        r"\[\[([^\]|#]+)(?:#[^\]|]*)?(?:\|([^\]]+))?\]\]",
        lambda m: normalize_title(m.group(2) if m.group(2) else m.group(1)),
        text,
    )

    text = re.sub(r"\[https?://[^\s\]]+\s+([^\]]+)\]", r"\1", text)
    text = re.sub(r"\[https?://[^\]]+\]", " ", text)
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    text = remove_templates(text)
    text = text.replace(chr(39) * 3, "").replace(chr(39) * 2, "")
    text = re.sub(r"={2,}[^=]+={2,}", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


characters = sorted({normalize_title(name) for name in pages_df["character_name"].dropna() if normalize_title(name)})
character_set = set(characters)

# Fast matcher for page-title character names. This avoids scanning every
# character name with a separate regex for every sentence.
def build_character_pattern(names: list[str]) -> re.Pattern:
    names = [name for name in names if len(name) >= 3]
    names = sorted(names, key=len, reverse=True)
    if not names:
        return re.compile(r"a^")  # matches nothing
    return re.compile(r"(?<![A-Za-z])(" + "|".join(re.escape(name) for name in names) + r")(?![A-Za-z])", flags=re.IGNORECASE)

CHARACTER_PATTERN = build_character_pattern(characters)
CHARACTER_LOOKUP = {name.casefold(): name for name in characters}

# Store a simple gazetteer for later inspection.
gazetteer_df = pd.DataFrame({"character_name": characters})
gazetteer_df.to_csv(CHARACTER_GAZETTEER_CSV, index=False)

print(f"Characters in gazetteer: {len(gazetteer_df):,}")
display(gazetteer_df.head(10))

Characters in gazetteer: 257


,character_name
0,Achim
1,Adelbert
2,Adolphine
3,Adrett
4,Aeussewahl
5,Albsenti
6,Alexis
7,Anastasius
8,Andrea
9,Angelica


## 5. Extract weak relationship signals from infoboxes

The MediaWiki character template often contains family fields such as `family/Father`, `family/Sister`, or `family/Spouse`. These provide useful weak labels.

This baseline maps:

- `family/Spouse`, `wife`, `husband`, `betrothed`, etc. → `romantic`
- other `family/...` fields → `family`

Only linked character names that also appear in the exported page-title gazetteer are kept.

In [5]:
def extract_balanced_template(wikitext: str, template_name: str = "Character") -> str:
    """Return the first balanced {{Character ...}} template block, if found."""
    if not isinstance(wikitext, str):
        return ""
    match = re.search(r"\{\{\s*" + re.escape(template_name) + r"\b", wikitext, flags=re.IGNORECASE)
    if not match:
        return ""

    start = match.start()
    i = start
    depth = 0
    while i < len(wikitext) - 1:
        two = wikitext[i : i + 2]
        if two == "{{":
            depth += 1
            i += 2
            continue
        if two == "}}":
            depth -= 1
            i += 2
            if depth == 0:
                return wikitext[start:i]
            continue
        i += 1
    return ""


def parse_template_fields(template_text: str) -> dict:
    """Parse simple |key=value lines from a template.
    """
    fields = {}
    current_key = None
    for raw_line in template_text.splitlines():
        line = raw_line.strip()
        if line.startswith("|") and "=" in line:
            key, value = line[1:].split("=", 1)
            current_key = key.strip()
            fields[current_key] = value.strip()
        elif current_key and line and not line.startswith("}}"):  # continuation line
            fields[current_key] += " " + line
    return fields


def relation_from_infobox_field(key: str, value: str) -> str | None:
    """Map an infobox field key/value pair to one of the relationship labels."""
    key_l = key.lower().strip()
    value_l = value.lower().strip()

    romantic_markers = [
        "spouse",
        "wife",
        "husband",
        "lover",
        "fiance",
        "fiancée",
        "betrothed",
    ]

    service_retainer_markers = [
        "retainer",
        "retainers",
        "attendant",
        "attendants",
        "guard knight",
        "guard knights",
        "scholar",
        "scholars",
        "serves",
        "served",
        "serving",
    ]

    if any(marker in key_l for marker in romantic_markers):
        return "romantic" if "romantic" in RELATIONSHIPS else None

    if key_l.startswith("family/") or key_l in {"familytree", "relatives"}:
        return "family" if "family" in RELATIONSHIPS else None

    if any(marker in key_l for marker in service_retainer_markers):
        return "service_retainer" if "service_retainer" in RELATIONSHIPS else None

    if key_l == "occupation" and any(marker in value_l for marker in service_retainer_markers):
        return "service_retainer" if "service_retainer" in RELATIONSHIPS else None

    return None


def extract_infobox_relation_edges(row: pd.Series) -> list[dict]:
    """Extract weakly labeled relation examples from one page's Character infobox."""
    head = row["character_name"]
    template = extract_balanced_template(row["character_text"], "Character")
    fields = parse_template_fields(template)
    examples = []

    for key, value in fields.items():
        label = relation_from_infobox_field(key, value)
        if label is None:
            continue

        for target, display_text in extract_wikilinks(value):
            if target in character_set and target != head:
                evidence = clean_wikitext(value)
                examples.append(
                    {
                        "head": head,
                        "tail": target,
                        "context": f"Infobox field {key}: {evidence}",
                        "section": "infobox",
                        "source_type": "infobox",
                        "weak_label": label,
                        "weak_label_source": f"infobox:{key}",
                    }
                )
    return examples


infobox_examples = []
for _, row in pages_df.iterrows():
    infobox_examples.extend(extract_infobox_relation_edges(row))

infobox_df = pd.DataFrame(infobox_examples)
print(f"Infobox weak relation examples: {len(infobox_df):,}")
if len(infobox_df):
    display(infobox_df.head(10))
    display(infobox_df["weak_label"].value_counts())

Infobox weak relation examples: 1,177


,head,tail,context,section,source_type,weak_label,weak_label_source
0,Adelbert,Bonifatius,Infobox field family/Brother: Bonifatius Bezewanst (In-Law),infobox,infobox,family,infobox:family/Brother
1,Adelbert,Bezewanst,Infobox field family/Brother: Bonifatius Bezewanst (In-Law),infobox,infobox,family,infobox:family/Brother
2,Adelbert,Irmhilde,"Infobox field family/Sister: Irmhilde (Half-Sister, Deceased) Unnamed Woman (Half-Sister, Deceased)",infobox,infobox,family,infobox:family/Sister
3,Adelbert,Veronica,"Infobox field family/Spouse: Veronica (First Wife) Irmhilde (Engaged, Deceased)",infobox,infobox,romantic,infobox:family/Spouse
4,Adelbert,Irmhilde,"Infobox field family/Spouse: Veronica (First Wife) Irmhilde (Engaged, Deceased)",infobox,infobox,romantic,infobox:family/Spouse
5,Adelbert,Constanze,Infobox field family/Daughter: Constanze Georgine Florencia (In-Law),infobox,infobox,family,infobox:family/Daughter
6,Adelbert,Georgine,Infobox field family/Daughter: Constanze Georgine Florencia (In-Law),infobox,infobox,family,infobox:family/Daughter
7,Adelbert,Florencia,Infobox field family/Daughter: Constanze Georgine Florencia (In-Law),infobox,infobox,family,infobox:family/Daughter
8,Adelbert,Sylvester,Infobox field family/Son: Sylvester Ferdinand Aub Frenbeltag (In-Law) Gieselfried (In-Law),infobox,infobox,family,infobox:family/Son
9,Adelbert,Ferdinand,Infobox field family/Son: Sylvester Ferdinand Aub Frenbeltag (In-Law) Gieselfried (In-Law),infobox,infobox,family,infobox:family/Son


weak_label
family              998
service_retainer     93
romantic             86
Name: count, dtype: int64

## 6. Generate sentence-level candidate pairs

For each character page, this cell:

1. cleans the page text,
2. splits it into sentence-like contexts,
3. finds mentions of other exported character-page titles,
4. creates a candidate pair `(page character, mentioned character)`, and
5. assigns a weak label using relationship keywords.

These labels are noisy and should be treated as baseline/distant-supervision labels.

In [6]:
RELATION_PATTERNS = {
    "family": [
        r"\bfather\b", r"\bmother\b", r"\bparent\b", r"\bparents\b", r"\bson\b", r"\bdaughter\b",
        r"\bsibling\b", r"\bbrother\b", r"\bsister\b", r"\buncle\b", r"\baunt\b", r"\bcousin\b",
        r"\bgrandfather\b", r"\bgrandmother\b", r"\brelative\b", r"\bin-law\b", r"\bniece\b", r"\bnephew\b",
    ],
    "romantic": [
        r"\bwife\b", r"\bhusband\b", r"\bspouse\b", r"\bmarried\b", r"\bmarriage\b",
        r"\bengaged\b", r"\bengagement\b", r"\bbetrothed\b", r"\bfianc[eé]\b",
        r"\blover\b", r"\blove interest\b", r"\bin love\b", r"\bromantic\b",
    ],
    "friend_ally": [
        r"\bfriend\b", r"\bfriends\b", r"\bfriendship\b",
        r"\bally\b", r"\ballies\b", r"\ballied\b", r"\bcompanion\b",
        r"\bclose friend\b", r"\bbest friend\b", r"\btrusted friend\b", r"\bconfidant\b",
    ],
    "service_retainer": [
        r"\bretainer\b", r"\bretainers\b", r"\bguard knight\b", r"\bguard knights\b",
        r"\battendant\b", r"\battendants\b",
        r"\bscholar\b", r"\bscholars\b",
        r"\bserved\b", r"\bserves\b", r"\bserving\b",
        r"\bin .* service\b", r"\bhead attendant\b", r"\bhead scholar\b", r"\bhead guard knight\b",
    ],
    "enemy_rival": [
        r"\benemy\b", r"\benemies\b", r"\brival\b", r"\brivals\b", r"\bopponent\b", r"\bopposes\b",
        r"\bbetray\b", r"\bbetrayed\b", r"\bkill(?:ed|s)?\b", r"\bexecute(?:d|s)?\b", r"\bhate(?:d|s)?\b",
        r"\babuse(?:d|s)?\b", r"\bjealous\b", r"\bpersecution\b", r"\battack(?:ed|s)?\b",
    ],
}

# Keep only patterns for labels present in RELATIONSHIPS.
RELATION_PATTERNS = {label: pats for label, pats in RELATION_PATTERNS.items() if label in RELATIONSHIPS}
RELATION_PRIORITY = ["romantic", "family", "enemy_rival", "service_retainer", "friend_ally"]
RELATION_PRIORITY = [label for label in RELATION_PRIORITY if label in RELATIONSHIPS]


def split_wikitext_into_sections(wikitext: str) -> list[tuple[str, str]]:
    """Split raw wikitext by Fandom-style {{h1|...}} / {{h2|...}} headings."""
    if not isinstance(wikitext, str):
        return [("lead", "")]

    heading_re = re.compile(r"\{\{h[12]\|([^}|]+)(?:\|[^}]*)?\}\}", flags=re.IGNORECASE | re.DOTALL)
    sections = []
    current_section = "lead"
    last = 0

    for match in heading_re.finditer(wikitext):
        if match.start() > last:
            sections.append((current_section, wikitext[last: match.start()]))
        current_section = clean_wikitext(match.group(1)) or "section"
        last = match.end()

    sections.append((current_section, wikitext[last:]))
    return sections


def split_sentences(text: str) -> list[str]:
    """Simple sentence splitter for cleaned wiki text."""
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    pieces = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9'\"“])", text)
    return [piece.strip() for piece in pieces if len(piece.strip()) >= MIN_CONTEXT_CHARS]



def find_mentioned_characters(text: str, head: str) -> list[str]:
    """Find other character names from the gazetteer mentioned in text."""
    found = []
    for match in CHARACTER_PATTERN.finditer(text):
        canonical = CHARACTER_LOOKUP.get(match.group(0).casefold())
        if canonical and canonical != head:
            found.append(canonical)
    return sorted(set(found))


def infer_weak_label_from_text(text: str, section: str = "") -> tuple[str, str]:
    """Infer a weak label from keyword patterns."""
    full_text = f"{section} {text}".lower()
    scores = {}
    matches = {}

    for label, patterns in RELATION_PATTERNS.items():
        label_matches = [pat for pat in patterns if re.search(pat, full_text, flags=re.IGNORECASE)]
        if label_matches:
            scores[label] = len(label_matches)
            matches[label] = label_matches

    if not scores:
        return NO_RELATION_LABEL, "keyword:none"

    # Highest score wins. Priority order breaks ties.
    max_score = max(scores.values())
    best_labels = [label for label, score in scores.items() if score == max_score]
    for label in RELATION_PRIORITY:
        if label in best_labels:
            return label, f"keyword:{','.join(matches[label][:3])}"
    return best_labels[0], f"keyword:{','.join(matches[best_labels[0]][:3])}"


def mark_entities(context: str, head: str, tail: str) -> str:
    """Add explicit entity markers for the target pair."""
    marked = context

    tail_pattern = re.compile(r"(?<![A-Za-z])" + re.escape(tail) + r"(?![A-Za-z])", flags=re.IGNORECASE)
    head_pattern = re.compile(r"(?<![A-Za-z])" + re.escape(head) + r"(?![A-Za-z])", flags=re.IGNORECASE)

    marked = tail_pattern.sub(lambda m: f"[TAIL] {m.group(0)} [/TAIL]", marked, count=1)

    if head_pattern.search(marked):
        marked = head_pattern.sub(lambda m: f"[HEAD] {m.group(0)} [/HEAD]", marked, count=1)
    else:
        marked = f"[HEAD] {head} [/HEAD] {marked}"

    return marked


def make_model_text(row: pd.Series, use_markers: bool = True, include_metadata: bool = True) -> str:
    """Build the text input for TF-IDF."""
    context = row["context"]
    if use_markers:
        context = mark_entities(context, row["head"], row["tail"])
    if include_metadata:
        return f"section={row['section']} source={row['source_type']} {context}"
    return context


def generate_sentence_candidates(pages: pd.DataFrame) -> pd.DataFrame:
    examples = []

    for _, row in pages.iterrows():
        head = row["character_name"]
        for section, raw_section in split_wikitext_into_sections(row["character_text"]):
            cleaned_section = clean_wikitext(raw_section)
            for sentence in split_sentences(cleaned_section):
                mentioned = find_mentioned_characters(sentence, head)
                for tail in mentioned:
                    weak_label, weak_source = infer_weak_label_from_text(sentence, section)
                    examples.append(
                        {
                            "head": head,
                            "tail": tail,
                            "context": sentence,
                            "section": section,
                            "source_type": "prose",
                            "weak_label": weak_label,
                            "weak_label_source": weak_source,
                        }
                    )

    return pd.DataFrame(examples)


sentence_df = generate_sentence_candidates(pages_df)
print(f"Sentence-level candidate examples: {len(sentence_df):,}")
if len(sentence_df):
    display(sentence_df.head(10))
    display(sentence_df["weak_label"].value_counts())

Sentence-level candidate examples: 4,858


,head,tail,context,section,source_type,weak_label,weak_label_source
0,Achim,Egon,"Alongside Egon, a fellow priest, Achim is assigned to spend a winter in Hasse teaching the local commoners how to interact with nobles.",lead,prose,no_relation,keyword:none
1,Achim,Rozemyne,"Later, he assists Rozemyne and her Gutenbergs in establishing new paper-making workshops in the provinces of Ehrenfest.",lead,prose,no_relation,keyword:none
2,Achim,Rozemyne,"After the former mayor of Hasse is executed for treason against the archduke, Rozemyne is determined to educate the local commoners to p...",Part 3 Volume 3,prose,enemy_rival,keyword:\bexecute(?:d|s)?\b
3,Achim,Rozemyne,"She is especially worried about Richt, the new mayor, who seems to have little knowledge of noble euphemisms. (For example, he offers Ro...",Part 3 Volume 4,prose,no_relation,keyword:none
4,Achim,Egon,"In the autumn following Hasse's first year of punishment, she assigns the gray priests Achim and Egon to stay in the village over the wi...",Part 3 Volume 5,prose,no_relation,keyword:none
5,Achim,Rozemyne,"While in Hasse, the priests are also tasked with gathering local folktales for ""Operation Grimm,"" which Rozemyne plans to one day compil...",Part 3 Volume 5,prose,no_relation,keyword:none
6,Achim,Egon,Although Achim and Egon initially struggle to adjust to the unclean living conditions of Hasse's winter mansion and the rough table mann...,Part 3 Volume 5,prose,no_relation,keyword:none
7,Achim,Rozemyne,"When Rozemyne returns from her first term at the Royal Academy, she once again assigns Achim to travel to the provinces, this time as pa...",Part 4 Volume 3,prose,no_relation,keyword:none
8,Adelbert,Sylvester,"After his passing, leadership of the duchy passed to his eldest son and heir, Sylvester.",lead,prose,family,keyword:\bson\b
9,Adelbert,Constanze,"With his first wife Veronica, Adelbert had three children: Georgine, Constanze and Sylvester.",lead,prose,romantic,keyword:\bwife\b


weak_label
no_relation         2577
family               900
romantic             616
service_retainer     531
enemy_rival          163
friend_ally           71
Name: count, dtype: int64

## 7. Build the initial candidate dataset

This cell creates `data/candidate_examples.csv`. The file contains every candidate pair used by all models, along with an initial weak `label` copied from `weak_label`.

If the optional LLM Judge is enabled in Section 8, the notebook will read this shared candidate file, generate `data/candidate_examples_llm_labeled.csv`, and replace `candidates_df` with the LLM-labeled version before training.


In [7]:
candidate_parts = []
if len(infobox_df):
    candidate_parts.append(infobox_df)
if len(sentence_df):
    candidate_parts.append(sentence_df)

if not candidate_parts:
    raise ValueError("No candidate examples were generated. Check the XML path and character-page filter.")

candidates_df = pd.concat(candidate_parts, ignore_index=True)
candidates_df = candidates_df.drop_duplicates(subset=["head", "tail", "context", "source_type"]).reset_index(drop=True)

# Keep only labels included in the configured relationship list.
candidates_df = candidates_df[candidates_df["weak_label"].isin(RELATIONSHIPS)].copy()

# Balance negatives against positives so the model does not learn to predict only no_relation.
pos_df = candidates_df[candidates_df["weak_label"] != NO_RELATION_LABEL].copy()
neg_df = candidates_df[candidates_df["weak_label"] == NO_RELATION_LABEL].copy()

if len(pos_df) == 0:
    raise ValueError("No positive relationship examples were generated. Add more relation keywords or enable an external labeling workflow.")

max_negatives = int(MAX_NEGATIVE_RATIO * len(pos_df))
if len(neg_df) > max_negatives:
    neg_df = neg_df.sample(n=max_negatives, random_state=RANDOM_SEED)

candidates_df = pd.concat([pos_df, neg_df], ignore_index=True)
candidates_df = candidates_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

candidates_df["label"] = candidates_df["weak_label"]
candidates_df["pair_id"] = candidates_df["head"] + " || " + candidates_df["tail"]
candidates_df["candidate_id"] = [f"cand_{i:06d}" for i in range(len(candidates_df))]
candidates_df["text_basic"] = candidates_df["context"]
candidates_df["text_marked"] = candidates_df.apply(lambda row: make_model_text(row, use_markers=True, include_metadata=True), axis=1)

candidates_df.to_csv(CANDIDATE_EXAMPLES_CSV, index=False)

label_counts = candidates_df["label"].value_counts().rename_axis("label").reset_index(name="count")
label_counts.to_csv(WEAK_LABEL_DISTRIBUTION_CSV, index=False)

print(f"Training candidates after balancing: {len(candidates_df):,}")
display(label_counts)
display(candidates_df[["candidate_id", "head", "tail", "label", "source_type", "context"]].head(10))

Training candidates after balancing: 6,031


,label,count
0,no_relation,2577
1,family,1897
2,romantic,702
3,service_retainer,621
4,enemy_rival,163
5,friend_ally,71


,candidate_id,head,tail,label,source_type,context
0,cand_000000,Georgine,Sylvester,family,prose,"She is the mother of Detlinde, Sylvester's older sister and a former archduke candidate of Ehrenfest."
1,cand_000001,Magdalena,Werdekraf,romantic,prose,"She also often trained with her brother Werdekraf before she moved to live at the Royal Palace, following her marriage."
2,cand_000002,Arno,Ferdinand,enemy_rival,prose,"Ferdinand, displeased at Arno's actions, had Arno killed."
3,cand_000003,Magdalena,Hildebrand,no_relation,prose,The boy is Hildebrand and was raised to become a vassal to whichever of his two older brothers would win the position of successor.
4,cand_000004,Wilfried,Veronica,family,prose,Upon Veronica's imprisonment Wilfried was told that his grandmother had fallen ill and had been moved to a far away place to recover.
5,cand_000005,Lasfam,Ferdinand,service_retainer,prose,"When Ferdinand personally asked him to do as the others did so Lasfam wouldn't have to suffer so much, Lasfam refused, stating that doin..."
6,cand_000006,Eckhart,Lamprecht,family,prose,"Despite despising Veroncia as well, his father still didn't want this to come to pass and when he learned of Eckhart's plans intervened ..."
7,cand_000007,Lungtase,Raufereg,family,prose,"Her less well behaved brother Raufereg has to stay home, since his parents don't want to risk embarrassing themselves and by extension t..."
8,cand_000008,Gloria,Rozemyne,no_relation,prose,"When Rozemyne enters noble society, Gloria holds her in contempt and often slanders her reputation at tea parties and social gatherings."
9,cand_000009,Georgine,Veronica,service_retainer,prose,"While Veronica was frozen in terror, one of her attendants had immediately sprung into action and administered an antidote."


## 8. Select labels for train/dev/test

The LLM Judge is intentionally not executed in this notebook. If `data/candidate_examples_llm_labeled.csv` exists from `01.5_llm_judge_colab.ipynb`, this notebook can consume it and split those labels. Otherwise it uses the weak labels from `data/candidate_examples.csv`.


In [8]:
def validate_candidate_training_frame(df: pd.DataFrame, frame_name: str = "candidates_df") -> pd.DataFrame:
    """Validate the candidate DataFrame used by the downstream training pipeline."""
    required_cols = {
        "candidate_id",
        "head",
        "tail",
        "context",
        "source_type",
        "weak_label",
        "label",
        "pair_id",
        "text_basic",
        "text_marked",
    }
    missing_cols = sorted(required_cols - set(df.columns))
    if missing_cols:
        raise ValueError(f"{frame_name} is missing required columns: {missing_cols}")

    invalid_labels = sorted(set(df["label"].dropna().astype(str)) - set(RELATIONSHIPS))
    if invalid_labels:
        raise ValueError(f"{frame_name} contains labels not in RELATIONSHIPS: {invalid_labels}")

    if df["label"].isna().any():
        raise ValueError(f"{frame_name} contains missing labels.")

    return df.reset_index(drop=True).copy()


def load_training_candidates(label_source_mode: str = LABEL_SOURCE_MODE) -> tuple[pd.DataFrame, str, Path]:
    """Load weak labels or externally generated LLM Judge labels without running an LLM locally."""
    mode = str(label_source_mode).strip().lower()
    if mode not in {"auto", "weak_labels", "llm_judge"}:
        raise ValueError("LABEL_SOURCE_MODE must be one of: 'auto', 'weak_labels', 'llm_judge'.")

    if mode == "llm_judge":
        if not LLM_LABELED_CANDIDATES_CSV.exists():
            raise FileNotFoundError(
                f"LABEL_SOURCE_MODE='llm_judge' but {LLM_LABELED_CANDIDATES_CSV} does not exist. "
                "Run 01_llm_judge_colab_gpu_minimal.ipynb in Colab first, or set LABEL_SOURCE_MODE='weak_labels'."
            )
        source_csv = LLM_LABELED_CANDIDATES_CSV
        label_source = "llm_judge"
    elif mode == "auto" and LLM_LABELED_CANDIDATES_CSV.exists():
        source_csv = LLM_LABELED_CANDIDATES_CSV
        label_source = "llm_judge"
    else:
        source_csv = CANDIDATE_EXAMPLES_CSV
        label_source = "weak_labels"

    df = pd.read_csv(source_csv)
    df = validate_candidate_training_frame(df, frame_name=f"{label_source} candidates")
    return df, label_source, source_csv


candidates_df, LABEL_SOURCE, CANDIDATE_SOURCE_CSV = load_training_candidates()

training_label_counts = candidates_df["label"].value_counts().rename_axis("label").reset_index(name="count")
training_label_counts.to_csv(TRAINING_LABEL_DISTRIBUTION_CSV, index=False)

print(f"Label source for train/dev/test: {LABEL_SOURCE}")
print(f"Candidate source CSV: {CANDIDATE_SOURCE_CSV}")
display(training_label_counts)


Label source for train/dev/test: llm_judge
Candidate source CSV: data\candidate_examples_llm_labeled.csv


,label,count
0,family,1940
1,no_relation,1366
2,romantic,964
3,friend_ally,687
4,service_retainer,649
5,enemy_rival,425


## 9. Train/dev/test split by character pair

The split uses `pair_id = head || tail` as the group. This reduces leakage, because the same character pair should not appear in both train and test.

In [9]:
def group_split_dataframe(df: pd.DataFrame, group_col: str = "pair_id"):
    """Split into train/dev/test using grouped splits.

    If the grouped split fails due to very small data, fall back to stratified random splits.
    """
    df = df.copy().reset_index(drop=True)
    groups = df[group_col]

    try:
        splitter1 = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
        train_dev_idx, test_idx = next(splitter1.split(df, groups=groups))
        train_dev_df = df.iloc[train_dev_idx].copy()
        test_df = df.iloc[test_idx].copy()

        relative_dev_size = DEV_SIZE / (1.0 - TEST_SIZE)
        splitter2 = GroupShuffleSplit(n_splits=1, test_size=relative_dev_size, random_state=RANDOM_SEED)
        train_idx, dev_idx = next(splitter2.split(train_dev_df, groups=train_dev_df[group_col]))
        train_df = train_dev_df.iloc[train_idx].copy()
        dev_df = train_dev_df.iloc[dev_idx].copy()
        split_method = "grouped_by_pair"
    except Exception as exc:
        print(f"Grouped split failed ({exc}). Falling back to stratified random split.")
        train_dev_df, test_df = train_test_split(
            df,
            test_size=TEST_SIZE,
            random_state=RANDOM_SEED,
            stratify=df["label"] if df["label"].nunique() > 1 else None,
        )
        relative_dev_size = DEV_SIZE / (1.0 - TEST_SIZE)
        train_df, dev_df = train_test_split(
            train_dev_df,
            test_size=relative_dev_size,
            random_state=RANDOM_SEED,
            stratify=train_dev_df["label"] if train_dev_df["label"].nunique() > 1 else None,
        )
        split_method = "stratified_random"

    return (
        train_df.reset_index(drop=True),
        dev_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
        split_method,
    )


train_df, dev_df, test_df, split_method = group_split_dataframe(candidates_df)

if train_df["label"].nunique() < 2:
    raise ValueError("Training split has fewer than two classes. Add more labels/examples or reduce filtering.")

train_df.to_csv(TRAIN_CSV, index=False)
dev_df.to_csv(DEV_CSV, index=False)
test_df.to_csv(TEST_CSV, index=False)

print(f"Split method: {split_method}")
print(f"Train: {len(train_df):,} | Dev: {len(dev_df):,} | Test: {len(test_df):,}")

split_summary = pd.concat(
    [
        train_df["label"].value_counts().rename("train"),
        dev_df["label"].value_counts().rename("dev"),
        test_df["label"].value_counts().rename("test"),
    ],
    axis=1,
).fillna(0).astype(int)
split_summary.to_csv(SPLIT_LABEL_DISTRIBUTION_CSV)
display(split_summary)

Split method: grouped_by_pair
Train: 4,251 | Dev: 915 | Test: 865


,train,dev,test
label,,,
family,1393,275,272
no_relation,936,221,209
romantic,688,151,125
service_retainer,471,94,84
friend_ally,467,108,112
enemy_rival,296,66,63


## Data pipeline metadata

This small metadata file lets the downstream split notebooks restore the correct label source and split method without relying only on notebook state.


In [10]:
pipeline_metadata = {
    "label_source": LABEL_SOURCE,
    "candidate_source_csv": str(CANDIDATE_SOURCE_CSV),
    "weak_candidate_examples_csv": str(CANDIDATE_EXAMPLES_CSV),
    "llm_labeled_candidates_csv": str(LLM_LABELED_CANDIDATES_CSV),
    "label_source_mode": LABEL_SOURCE_MODE,
    "split_method": split_method,
    "train_csv": str(TRAIN_CSV),
    "dev_csv": str(DEV_CSV),
    "test_csv": str(TEST_CSV),
}
PIPELINE_RUN_METADATA_JSON.write_text(json.dumps(pipeline_metadata, indent=2), encoding="utf-8")
print(json.dumps(pipeline_metadata, indent=2))


{
  "label_source": "llm_judge",
  "candidate_source_csv": "data\\candidate_examples_llm_labeled.csv",
  "weak_candidate_examples_csv": "data\\candidate_examples.csv",
  "llm_labeled_candidates_csv": "data\\candidate_examples_llm_labeled.csv",
  "label_source_mode": "auto",
  "split_method": "grouped_by_pair",
  "train_csv": "data\\train.csv",
  "dev_csv": "data\\dev.csv",
  "test_csv": "data\\test.csv"
}
